<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/t5_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers "datasets<3" sentencepiece accelerate evaluate

In [ ]:
import re
import torch
import pandas as pd

from datasets import load_dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to {device}")

torch.manual_seed(0)

# NQ dataset

In [ ]:
nq = load_dataset("sentence-transformers/natural-questions", split="train[:5000]")
split_dataset = nq.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]
dataset_name = "natural-questions"

# Helper functions

In [ ]:
def get_question(example):
  return example["query"]

def get_answer(example):
  answer = example["answer"]
  if isinstance(answer, list):
    answer = answer[0]
  return answer

In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

# Load T5

In [ ]:
model_name = "t5-base"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

print("Loaded model: ", model_name)

# Converting nq into T5 nput format

In [ ]:
def make_input(question):
  return f"question: {question}"

# Creating examples to train/evaluate

In [ ]:
def make_examples(dataset):
  examples = []
  for i in range(len(dataset)):
    question = get_question(dataset[i])
    answer = get_answer(dataset[i])
    examples.append(
        "input_text": make_input(question),
        "target_text": answer,
        "question": question
    )
  return pd.DataFrame(examples)

train_df = make_examples(train_dataset)
eval_df = make_examples(eval_dataset)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

# Preprocessing

In [ ]:
def preprocess_function(examples):
  inputs = tokenizer(examples["input_text"], padding="max_length", truncation=True, max_length=128)
  labels = tokenizer(text_target = examples["target_text"], padding="max_length", truncation=True, max_length=32)
  inputs["labels"] = labels["input_ids"]
  return inputs

# Tokenize

In [ ]:
tokenized_train_dataset = train_dataset.map(preprocess_function, remove_columns=train_dataset.column_names)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, remove_columns=eval_dataset.column_names)

# Data collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# EM metrics

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_pred = tokenizer.batch_decode(predictions, skip_special_tokens = True)
    labels = [[token if token != -100 else tokenizer.pad_token_id for token in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens = True)

    em = [int(normalize_text(pred) == normalize_text(label)) for pred, label in zip(decoded_pred, decoded_labels)]

    return {"em": 100 * sum(em) / len(em)}

# Training arguments

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    logging_strategy="steps",
    logging_steps=100,
    predict_with_generate=True,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none"
)

# Trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Evaluate

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
predictions = trainer.predict(tokenized_eval_dataset)

predictions, labels = predictions.predictions, predictions.label_ids
decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)

labels = [[token if token != -100 else tokenizer.pad_token_id for token in label] for label in labels]
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

# Saving results

In [ ]:
results_df = pd.DataFrame({
    "question": eval_df["question"],
    "answer": decoded_labels,
    "prediction": decoded_predictions
})
results_df["em"]=[
    int(normalize_text(pred) == normalize_text(label))
    for pred, label in zip(results_df["prediction"], results_df["answer"])
]
results_df.to_csv("t5_results.csv", index=False)
print("Saved t5_results.csv")